In [3]:
# libraries

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler 

In [2]:
## load up the blood data

OPT_blood = pd.read_csv('/home/aabdulrasul/Projects/baard/temp/processed/baseline_blood.csv')

In [ ]:



## we need to make the SASP index, which is a PCA of all the blood markers we have. this is what was said in the paper: 

# We calculated a SASP index for each participant based on the regression analysis of individual weights of biomarkers included in the SASP panel. 
# For this, we initially carried out a principal component analysis (PCA) with all proteins included in the model. We extracted the individual weight 
# of each biomarker based on its eigenvector value. Finally, we calculated the SASP index for each participant using a multiple linear regression model, 
# in which SASP index was the dependent variable, the individual SASP biomarkers were the predictor variables, and the biomarker’s weight was the regression coefficient for each SASP biomarker:


# select the blood markers
blood_markers = OPT_blood[['IL-6','gp130','IL-8/CXCL8','uPAR','MIF','CCL2/JE/MCP-1',
                           'Osteoprotegerin/TNFRSF11B','IL-1 beta/IL-1F2','CCL20/MIP-3 alpha',
                           'CCL3/MIP-1 alpha','CCL4/MIP-1 beta','CCL13/MCP-4','GM-CSF',
                           'ICAM-1/CD54','TNF RII/TNFRSF1B','TNF RI/TNFRSF1A','PIGF',
                           'CXCL1/GRO alpha/KC/CINC-1','IGFBP-2','TIMP-1','IGFBP-6','Angiogenin']]


# 1) ORIGINAL (no log transform)

# Standardize
scaler_raw = StandardScaler()
blood_scaled_raw = scaler_raw.fit_transform(blood_markers)

# PCA
pca_raw = PCA(n_components=1)
pca_raw.fit(blood_scaled_raw)

# Extract eigenvector loadings (weights)
weights_raw = pca_raw.components_[0]

# Compute SASP index
sasp_index_raw = np.dot(blood_scaled_raw, weights_raw)

# save in dataframe
OPT_blood['SASP_index_raw'] = sasp_index_raw

# 2) LOG2-TRANSFORMED VERSION (paper method)

# Handle zeros: replace zeros with half the smallest non-zero value
blood_nonzero = blood_markers.replace(0, np.nan)
smallest = blood_nonzero.min().min()
blood_filled = blood_nonzero.fillna(smallest / 2)

# log2 transform
blood_log2 = np.log2(blood_filled)

# standardize
scaler_log = StandardScaler()
blood_scaled_log = scaler_log.fit_transform(blood_log2)

# PCA
pca_log = PCA(n_components=1)
pca_log.fit(blood_scaled_log)

# Extract weights
weights_log = pca_log.components_[0]

# Compute SASP index
sasp_index_log = np.dot(blood_scaled_log, weights_log)

# add to dataframe
OPT_blood['SASP_index_log2'] = sasp_index_log


OPT_blood.to_csv('/home/aabdulrasul/Projects/baard/code/OPT_blood_SASP.csv', index=False)



In [16]:
print("Variance explained by PC1 (%):", pca.explained_variance_ratio_[0] * 100)


Variance explained by PC1 (%): 19.14224325109128


np.float64(4.273466327551312)